<a href="https://colab.research.google.com/github/Harshithpalan/Python-projects/blob/main/%20product%20recommendation%20model%20based%20on%20user%20rating.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### 1. Data Acquisition
We'll download the MovieLens 1M dataset from the official GroupLens website.

In [ ]:
import pandas as pd
import os
import requests
import zipfile

# Download the dataset
url = 'https://files.grouplens.org/datasets/movielens/ml-1m.zip'
zip_path = 'ml-1m.zip'
extract_path = 'ml-1m'

print('Downloading MovieLens 1M...')
r = requests.get(url)
with open(zip_path, 'wb') as f:
    f.write(r.content)

print('Unzipping...')
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall('.')

print('Done.')

Unzipping...
Done.


### 2. Loading the Data
The files are formatted with `::` as a separator. We will load the `ratings.dat` and `movies.dat` files.

In [ ]:
# Load Ratings
# UserID::MovieID::Rating::Timestamp
ratings = pd.read_csv('ml-1m/ratings.dat', sep='::', engine='python',
                      names=['user_id', 'movie_id', 'rating', 'timestamp'])

# Load Movies
# MovieID::Title::Genres
movies = pd.read_csv('ml-1m/movies.dat', sep='::', engine='python',
                     names=['movie_id', 'title', 'genres'], encoding='latin-1')

print(f'Loaded {len(ratings)} ratings and {len(movies)} movies.')
display(ratings.head())
display(movies.head())

Loaded 1000209 ratings and 3883 movies.


,user_id,movie_id,rating,timestamp
0,1,1193,5,978300760
1,1,661,3,978302109
2,1,914,3,978301968
3,1,3408,4,978300275
4,1,2355,5,978824291


,movie_id,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


### 3. Data Preprocessing
We need to encode `user_id` and `movie_id` as continuous integers for the embedding layers and split the data into training and testing sets.

In [ ]:
from sklearn.model_selection import train_test_split

# Map user_id and movie_id to continuous indices
ratings['user_idx'] = ratings['user_id'].astype('category').cat.codes
ratings['movie_idx'] = ratings['movie_id'].astype('category').cat.codes

num_users = ratings['user_idx'].nunique()
num_movies = ratings['movie_idx'].nunique()

print(f'Number of unique users: {num_users}')
print(f'Number of unique movies: {num_movies}')

# Create train and test sets
train, test = train_test_split(ratings, test_size=0.2, random_state=42)

print(f'Training samples: {len(train)}')
print(f'Testing samples: {len(test)}')

Number of unique users: 6040
Number of unique movies: 3706
Training samples: 800167
Testing samples: 200042


### 4. Building the Recommender Model
We will use `tf.keras` to create a Dot Product model with embeddings.

In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models

class RecommenderNet(tf.keras.Model):
    def __init__(self, num_users, num_movies, embedding_size, **kwargs):
        super(RecommenderNet, self).__init__(**kwargs)
        self.user_embedding = layers.Embedding(
            num_users, embedding_size, embeddings_initializer='he_normal',
            embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
        )
        self.user_bias = layers.Embedding(num_users, 1)
        self.movie_embedding = layers.Embedding(
            num_movies, embedding_size, embeddings_initializer='he_normal',
            embeddings_regularizer=tf.keras.regularizers.l2(1e-6)
        )
        self.movie_bias = layers.Embedding(num_movies, 1)

    def call(self, inputs):
        user_vector = self.user_embedding(inputs[:, 0])
        user_bias = self.user_bias(inputs[:, 0])
        movie_vector = self.movie_embedding(inputs[:, 1])
        movie_bias = self.movie_bias(inputs[:, 1])
        dot_user_movie = tf.reduce_sum(tf.multiply(user_vector, movie_vector), axis=1, keepdims=True)
        x = dot_user_movie + user_bias + movie_bias
        return tf.nn.sigmoid(x) * 5 # Scale sigmoid to 0-5 range

EMBEDDING_SIZE = 50
model = RecommenderNet(num_users, num_movies, EMBEDDING_SIZE)
model.compile(
    loss=tf.keras.losses.MeanSquaredError(),
    optimizer=tf.keras.optimizers.Adam(learning_rate=0.001)
)

# Prepare inputs
x_train = train[['user_idx', 'movie_idx']].values
y_train = train['rating'].values.astype('float32')
x_test = test[['user_idx', 'movie_idx']].values
y_test = test['rating'].values.astype('float32')

print('Model compiled and data prepared.')

Model compiled and data prepared.


### 5. Training the Model
We'll train the model for a few epochs.

In [ ]:
history = model.fit(
    x=x_train,
    y=y_train,
    batch_size=64,
    epochs=5,
    validation_data=(x_test, y_test),
    verbose=1
)

Epoch 1/5
12503/12503 ━━━━━━━━━━━━━━━━━━━━ 112s 9ms/step - loss: 0.9888 - val_loss: 0.8004
Epoch 2/5
12503/12503 ━━━━━━━━━━━━━━━━━━━━ 113s 9ms/step - loss: 0.7238 - val_loss: 0.7578
Epoch 3/5
12503/12503 ━━━━━━━━━━━━━━━━━━━━ 108s 9ms/step - loss: 0.6239 - val_loss: 0.7681
Epoch 4/5
12503/12503 ━━━━━━━━━━━━━━━━━━━━ 140s 9ms/step - loss: 0.5450 - val_loss: 0.8027
Epoch 5/5
12503/12503 ━━━━━━━━━━━━━━━━━━━━ 141s 8ms/step - loss: 0.4938 - val_loss: 0.8422
